# EduMentor AI - Data Cleaning & Preprocessing

This notebook prepares a clean student-level dataset from OULAD for risk prediction.

In [1]:
from pathlib import Path
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
project_raw = Path('../data/raw')
external_raw = os.getenv('OULAD_DATASET_PATH', r'C:/Users/PC/Downloads/open+university+learning+analytics+dataset')
DATA_RAW = project_raw if project_raw.exists() and any(project_raw.iterdir()) else Path(external_raw)

DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

table_files = {
    'student_info': 'studentInfo.csv',
    'student_registration': 'studentRegistration.csv',
    'student_assessment': 'studentAssessment.csv',
    'assessments': 'assessments.csv',
    'student_vle': 'studentVle.csv',
    'courses': 'courses.csv',
}

tables = {}
for name, filename in table_files.items():
    fp = DATA_RAW / filename
    if fp.exists():
        tables[name] = pd.read_csv(fp)

if 'student_info' not in tables:
    raise FileNotFoundError(f'studentInfo.csv is required. Checked: {DATA_RAW}')

print(f'Using raw dataset path: {DATA_RAW}')
{k: v.shape for k, v in tables.items()}

Using raw dataset path: C:\Users\PC\Downloads\open+university+learning+analytics+dataset


{'student_info': (32593, 12),
 'student_registration': (32593, 5),
 'student_assessment': (173912, 5),
 'assessments': (206, 6),
 'student_vle': (10655280, 6),
 'courses': (22, 3)}

In [4]:
# Build student-level feature table
student_info = tables['student_info'].copy()
key_cols = ['code_module', 'code_presentation', 'id_student']
df = student_info.copy()

if 'student_registration' in tables:
    registration = tables['student_registration'].copy()
    registration['date_registration'] = pd.to_numeric(registration['date_registration'], errors='coerce')
    registration['date_unregistration'] = pd.to_numeric(registration['date_unregistration'], errors='coerce')

    reg_agg = registration.groupby(key_cols, dropna=False).agg(
        registration_day=('date_registration', 'mean'),
        unregister_day=('date_unregistration', 'mean'),
    ).reset_index()
    df = df.merge(reg_agg, on=key_cols, how='left')

if {'student_assessment', 'assessments'}.issubset(set(tables.keys())):
    assessment = tables['student_assessment'].copy()
    assessments = tables['assessments'].copy()

    assessment = assessment.merge(
        assessments[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'weight']],
        on='id_assessment',
        how='left',
    )

    assessment['score'] = pd.to_numeric(assessment['score'], errors='coerce')
    assessment['weight'] = pd.to_numeric(assessment['weight'], errors='coerce')

    ass_agg = assessment.groupby(key_cols, dropna=False).agg(
        avg_assessment_score=('score', 'mean'),
        submitted_assessments=('is_banked', 'count'),
        avg_assessment_weight=('weight', 'mean'),
    ).reset_index()
    df = df.merge(ass_agg, on=key_cols, how='left')

if 'student_vle' in tables:
    student_vle = tables['student_vle'].copy()
    student_vle['sum_click'] = pd.to_numeric(student_vle['sum_click'], errors='coerce')

    vle_agg = student_vle.groupby(key_cols, dropna=False).agg(
        total_clicks=('sum_click', 'sum'),
        avg_clicks_per_event=('sum_click', 'mean'),
        activity_events=('sum_click', 'count'),
    ).reset_index()
    df = df.merge(vle_agg, on=key_cols, how='left')

if 'final_result' not in df.columns:
    raise ValueError('final_result column not found in studentInfo.csv')

# Risk target: 1 = At Risk (Fail/Withdrawn), 0 = Safe
df['risk_label'] = df['final_result'].isin(['Fail', 'Withdrawn']).astype(int)
df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,final_result,registration_day,unregister_day,avg_assessment_score,submitted_assessments,avg_assessment_weight,total_clicks,avg_clicks_per_event,activity_events,risk_label
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,Pass,-159.0,NaN,82.0,5.0,20.0,934.0,4.765306,196.0,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,Pass,-53.0,NaN,66.4,5.0,20.0,1435.0,3.337209,430.0,0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,Withdrawn,-92.0,12.0,NaN,NaN,NaN,281.0,3.697368,76.0,1
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,Pass,-52.0,NaN,76.0,5.0,20.0,2158.0,3.254902,663.0,0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,Pass,-176.0,NaN,54.4,5.0,20.0,1034.0,2.937500,352.0,0


In [5]:
# Basic EDA checkpoints
missing_summary = df.isna().mean().sort_values(ascending=False).head(20)
class_distribution = df['risk_label'].value_counts(normalize=True)
print('Top missingness:')
print(missing_summary)
print('\nClass distribution (risk_label):')
print(class_distribution)

Top missingness:
unregister_day           0.690977
avg_assessment_score     0.207805
avg_assessment_weight    0.207100
submitted_assessments    0.207100
activity_events          0.103243
avg_clicks_per_event     0.103243
total_clicks             0.103243
registration_day         0.001381
code_module              0.000000
final_result             0.000000
disability               0.000000
code_presentation        0.000000
studied_credits          0.000000
num_of_prev_attempts     0.000000
age_band                 0.000000
imd_band                 0.000000
highest_education        0.000000
region                   0.000000
gender                   0.000000
id_student               0.000000
dtype: float64

Class distribution (risk_label):
risk_label
1    0.527966
0    0.472034
Name: proportion, dtype: float64


In [6]:
# Save cleaned dataset and train/test split for modeling
clean_path = DATA_PROCESSED / 'clean_data.csv'
df.to_csv(clean_path, index=False)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['risk_label']
)

train_df.to_csv(DATA_PROCESSED / 'train_data.csv', index=False)
test_df.to_csv(DATA_PROCESSED / 'test_data.csv', index=False)

print(f'Saved: {clean_path}')
print(f'Train rows: {len(train_df)} | Test rows: {len(test_df)}')

Saved: ..\data\processed\clean_data.csv
Train rows: 26074 | Test rows: 6519
